# ForgeAI on a Colab GPU
1. On your computer: `python colab/make_zip.py` builds `forgeai_colab.zip`.
2. **Runtime > Change runtime type > T4 GPU** (L4 or A100 are faster).
3. **Runtime > Run all.** Upload `forgeai_colab.zip` when asked, and allow the Google Drive prompt (it protects the run if Colab disconnects).
4. At **New brief**, type the brief and upload the documents.

If Colab disconnects, run all again: finished model calls are reused.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
import os
from google.colab import files
if not os.path.exists("/content/forgeai/pyproject.toml"):
    if not os.path.exists("/content/forgeai_colab.zip"):
        os.chdir("/content"); files.upload()  # choose forgeai_colab.zip
    !unzip -oq /content/forgeai_colab.zip -d /content/forgeai
os.chdir("/content/forgeai")
!ls

In [ ]:
import shutil, threading, time
from google.colab import drive
drive.mount("/content/drive")
BACKUP = "/content/drive/MyDrive/forgeai_runs"
if os.path.isdir(BACKUP):
    shutil.copytree(BACKUP, "runs", dirs_exist_ok=True)  # restore an interrupted session
def backup():
    while True:
        time.sleep(60)
        try:
            shutil.copytree("runs", BACKUP, dirs_exist_ok=True)
        except Exception as error:
            print("backup:", error)
threading.Thread(target=backup, daemon=True).start()

In [ ]:
!apt-get -qq update && apt-get -qq install -y zstd pciutils > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh
!pip install -q -e .

In [ ]:
import subprocess, urllib.request
vram = int(subprocess.check_output(["nvidia-smi", "--query-gpu=memory.total", "--format=csv,noheader,nounits"]).split()[0])
env = dict(os.environ, OLLAMA_CONTEXT_LENGTH="16384",  # the default is too small and would cut prompts silently
           OLLAMA_NUM_PARALLEL="4" if vram > 20000 else "2", OLLAMA_MAX_LOADED_MODELS="2",
           OLLAMA_FLASH_ATTENTION="1", OLLAMA_KV_CACHE_TYPE="q8_0", OLLAMA_KEEP_ALIVE="3h")
subprocess.Popen(["ollama", "serve"], env=env, stdout=open("/content/ollama.log", "w"), stderr=subprocess.STDOUT)
for _ in range(60):
    try:
        urllib.request.urlopen("http://127.0.0.1:11434/api/tags", timeout=2); break
    except Exception:
        time.sleep(1)
!ollama pull llama3.1:8b
!ollama pull qwen2.5:7b
os.environ["OLLAMA_API_KEY"] = "ollama"

In [ ]:
!forge doctor --config forge.colab.toml
!ollama ps  # PROCESSOR should say 100% GPU for both models

## New brief

In [ ]:
import shutil, time, glob
BRIEF = ""  #@param {type:"string"}
if not BRIEF:
    BRIEF = input("Paste the one-paragraph brief, then press Enter: ").strip()
RUN_ID = "review-" + time.strftime("%H%M%S")
docs = f"/content/review_docs/{RUN_ID}"
os.makedirs(docs, exist_ok=True)
print("Choose every document in the folder (or one .zip of it):")
os.chdir(docs); uploaded = files.upload(); os.chdir("/content/forgeai")
for name in uploaded:
    if name.endswith(".zip"):
        shutil.unpack_archive(f"{docs}/{name}", docs); os.remove(f"{docs}/{name}")
inner = [p for p in glob.glob(f"{docs}/*") if not os.path.basename(p).startswith((".", "__"))]
if len(inner) == 1 and os.path.isdir(inner[0]):  # a zip of a folder
    docs = inner[0]
shutil.copytree(docs, f"runs/{RUN_ID}-docs", dirs_exist_ok=True)  # keep the documents with the results
open("/content/brief.txt", "w").write(BRIEF)
print(f"If the build stops, continue with: !forge resume runs/{RUN_ID} --config forge.colab.toml")
!forge build --brief /content/brief.txt --docs runs/{RUN_ID}-docs --unattended --run-id {RUN_ID} --config forge.colab.toml

In [ ]:
!cat runs/{RUN_ID}/report.md
!zip -qr /content/{RUN_ID}.zip runs/{RUN_ID} runs/{RUN_ID}-docs
files.download(f"/content/{RUN_ID}.zip")

In [ ]:
# Ask the new agent one question (run again for another)
import subprocess
question = input("Question: ")
chat = ["forge", "chat", f"runs/{RUN_ID}/agent.final.yaml", "--config", "forge.colab.toml"]
print(subprocess.run(chat, input=question + "\n\n", capture_output=True, text=True).stdout)

## Demo runs (optional)

In [ ]:
RUN_DEMOS = False  # set True to rebuild the two committed demo runs (about an hour or more)

def build(name, brief, docs):
    if os.path.exists(f"runs/{name}/run.json"):
        command = f"resume runs/{name}"
    else:
        command = f"build --brief {brief} --docs {docs} --unattended --run-id {name}"
    !forge {command} --config forge.colab.toml

if RUN_DEMOS:
    build("mlc-crew-rights", "briefs/mlc.txt", "data/mlc")
    build("spares-procurement", "briefs/spares.txt", "data/spares")

In [ ]:
if RUN_DEMOS:
    !rm -f /content/forgeai_runs.zip && zip -qr /content/forgeai_runs.zip runs/mlc-crew-rights runs/spares-procurement
    shutil.copytree("runs", BACKUP, dirs_exist_ok=True)
    files.download("/content/forgeai_runs.zip")